# 08 · The link audit — and three findings this kills

## Why this notebook exists

`06` and `07` measured the company's monetisation surfaces by searching descriptions for specific
domain names: `lttstore.com` for the storefront, `floatplane.com` for the paywall,
`amzn.to|geni.us|howl.link|tag=` for affiliate links.

Then I looked at an actual recent description and found this line:

```
► GET EXCLUSIVE CONTENT ON FLOATPLANE: https://lmg.gg/lttfloatplane
```

The Floatplane link is there. It is not on `floatplane.com`. **Some time around early 2023 the
company moved its outbound links onto its own shortener, `lmg.gg`.** Every measurement built on
literal domain strings would read that migration as the link disappearing.

That is exactly what happened, and it produced two headline findings in `06`, one recommendation in
`07`, and one inference in `03` — all wrong.

This is a different failure from the composition errors elsewhere in the project. Those averaged
over a shifting mix. **This one measured a proxy and reported it as the thing.** The signal moved
while the underlying behaviour stayed constant.

The fix is not a better regex. It is to stop guessing which domains matter, **census every URL in
the corpus first**, and build the taxonomy from what is actually there.

## 0. Environment and data

In [ ]:
%pip install -q pandas numpy matplotlib

import json, re, pathlib, collections
import pandas as pd, numpy as np, matplotlib.pyplot as plt

CWD = pathlib.Path.cwd()
NB   = next((p for p in [CWD, CWD/"notebook", *CWD.parents] if (p/"data").is_dir()), CWD)
DATA, FIG = NB/"data", NB/"figures"; FIG.mkdir(exist_ok=True)

desc = (pd.DataFrame([json.loads(l) for l in (DATA/"content/descriptions.jsonl").open() if l.strip()])
          .drop_duplicates("id", keep="last"))
vids = pd.read_csv(DATA/"videos_clean.csv", parse_dates=["date"])
m = desc.merge(vids[["id","channel","date","views","format"]], on="id")
m["yr"]  = m.date.dt.year
m["txt"] = m.description.fillna("")
m["len"] = m.txt.str.len()
print(f"{len(m):,} descriptions · {m.date.min().date()} → {m.date.max().date()}")

## 1. Census first — what domains are actually in there?

No hypothesis, no regex list. Extract every host from every description and count how many videos
each appears in. Anything above a percent or two of the corpus is a surface I need a decision
about.

In [ ]:
URL = re.compile(r"https?://([A-Za-z0-9\.\-]+)")
hosts = collections.Counter()
for t in m.txt:
    for h in set(URL.findall(t)):
        hosts[h.lower().removeprefix("www.")] += 1

print(f"{len(hosts):,} distinct hosts across {len(m):,} descriptions\n")
print(f"{'videos':>7}  {'share':>6}  host")
for h, n in hosts.most_common(30):
    print(f"{n:>7,}  {n/len(m):>6.1%}  {h}")

> **`lmg.gg` is in 93.5% of every description in the corpus** — more than the storefront domain
> itself. It is a general-purpose wrapper, so the host alone says nothing about where a link goes.
> The *path* does: `lmg.gg/lttfloatplane`, `lmg.gg/partners`, `lmg.gg/sponsors`.
>
> Also visible: `piavpn.com` at 29%, `nexus.gg` at 11%, `affilimate.com` at 10%, `bit.ly` at 9%,
> then a long tail of individual sponsors — dbrand, Vessi, Ridge, Squarespace, Manscaped,
> DeleteMe. Not one of those was in the affiliate regex I used in `06`.

In [ ]:
paths = collections.Counter()
for t in m.txt:
    for p in re.findall(r"lmg\.gg/([A-Za-z0-9\-_]+)", t):
        paths[p.lower()] += 1
print("Most common lmg.gg paths\n")
for p, n in paths.most_common(18):
    print(f"  {n:>5,}  lmg.gg/{p}")

## 2. The decisive test — did Floatplane leave, or did the URL change?

Three ways of asking the same question. If the first collapses while the third holds, the finding
in `06` was an artefact and nothing about the company's behaviour changed.

In [ ]:
m["fp_domain"] = m.txt.str.contains(r"floatplane\.com", case=False, na=False)
m["fp_lmg"]    = m.txt.str.contains(r"lmg\.gg/\S*floatplane", case=False, na=False)
m["fp_any"]    = m.txt.str.contains(r"(?i)floatplane", na=False)

lv = m[m.format == "long_video"]
print("Long-form only, share of uploads (%)\n")
print((lv[lv.yr.between(2021, 2026)]
       .groupby("yr")[["fp_domain","fp_lmg","fp_any"]].mean()*100).round(1).to_string())

print("\n\nMonthly, across the supposed break\n")
q = lv[lv.date.between("2022-11-01","2023-06-30")].copy()
q["ym"] = q.date.dt.to_period("M")
print((q.groupby("ym")[["fp_domain","fp_lmg","fp_any"]].mean()*100).round(1).to_string())

> **Finding killed.**
>
> `06` reported that Floatplane links went from 89.5% of long-form uploads in January 2023 to 1.7%
> in February — a one-month cliff — and read it as the company withdrawing from its own paywall.
>
> Any mention of Floatplane: **99.1% in 2021, 98.6% in 2023, 100% in 2026.** It never moved. In
> January 2023 the domain form was at 89.5% and the shortener at 11.8%; in February the domain was
> at 1.7% and the shortener at 96.7%. **They changed the link, not the strategy.**
>
> **And this reverses an inference in `03`.** That notebook could not rule out the reading that
> falling YouTube reach was partly deliberate — value being moved behind the company's own paywall.
> `06` claimed to close that question, on the grounds that Floatplane promotion had been removed
> outright. It had not. **The paywall-migration hypothesis is fully open again**, and `03` was right
> to leave it open.

## 3. Same test on affiliate links

The regex in `06` was `amzn.to|geni.us|howl.link|tag=`. The census says that misses `piavpn.com`,
`nexus.gg`, `affilimate.com`, `bit.ly`, `kit.co`, the `lmg.gg/partners` and `lmg.gg/sponsors`
hubs, and every individual sponsor domain.

The broad pattern below is built from the census rather than from memory. A video counts as having
a commercial outbound link if it carries **any** of them.

In [ ]:
NARROW = r"amzn\.to|geni\.us|howl\.link|tag="

BROAD = "|".join([
    r"amzn\.to", r"geni\.us", r"howl\.link", r"tag=", r"affilimate\.com", r"nexus\.gg",
    r"kit\.co", r"bit\.ly", r"piavpn\.com", r"lmg\.gg/\S*(?:partner|sponsor)",
    r"shortlinus\.com", r"dbrand\.com", r"vessi\.com", r"ridge\.com", r"squarespace\.com",
    r"manscaped\.com", r"joindeleteme\.com", r"micro\.center", r"freshbooks\.com",
    r"odoo\.com", r"grammarly\.com", r"buildredux\.com", r"dro\.ps", r"prsm2\.com",
    r"playwt\.link", r"tm\.game",
])

m["aff_narrow"]  = m.txt.str.contains(NARROW, case=False, na=False, regex=True)
m["aff_broad"]   = m.txt.str.contains(BROAD,  case=False, na=False, regex=True)
m["store_broad"] = m.txt.str.contains(r"lttstore|lmg\.gg/\S*(?:store|merch)|shop our products",
                                      case=False, na=False)

lv = m[m.format == "long_video"]
print("Long-form, share of uploads (%)\n")
print((lv[lv.yr.between(2021, 2026)]
       .groupby("yr")[["aff_narrow","aff_broad","store_broad"]].mean()*100).round(1).to_string())

sc = lv[(lv.channel == "ShortCircuit") & (lv.yr >= 2024)]
print(f"\n\nShortCircuit, 2024 onward — {len(sc)} uploads")
print(f"  narrow definition: {sc.aff_narrow.mean():.1%}")
print(f"  broad  definition: {sc.aff_broad.mean():.1%}")
print(f"  uploads with no commercial outbound link at all: {(~sc.aff_broad).sum()}")

> **Second finding killed, and a recommendation with it.**
>
> Narrow definition: 96.8% in 2021 falling to 21.4% in 2026 — which is what `06` reported as
> affiliate links being withdrawn. Broad definition: **100% in 2021 and 100% in 2026.** Flat.
>
> The recommendation this generated was worse. I told the user that ShortCircuit — an unboxing
> channel, whose natural monetisation is "here is where to buy the thing I just opened" — had
> affiliate coverage of only 26%, and that closing the gap was worth about 57 million views of
> additional coverage. Under the broad definition ShortCircuit sits at **99.8%: one upload out of
> 471 carries no commercial outbound link.** There was never a gap. **Recommendation retracted.**
>
> Storefront attachment on long-form holds at 99% under both definitions, which is the one thing
> `06` got right about this.

## 4. What survives — and it is the biggest one

If the descriptions never stopped carrying links, the Shorts result should be an artefact too.
It is not, and the reason is visible in the raw text: **on Shorts the description stopped being a
link hub and became a caption.**

In [ ]:
sh = m[m.format == "short"]
print("Shorts descriptions by year\n")
print(sh[sh.yr.between(2021, 2026)].groupby("yr").apply(lambda g: pd.Series({
    "uploads":       len(g),
    "median_length": g["len"].median(),
    "empty_%":       100*(g["len"] < 5).mean(),
    "store_%":       100*g.store_broad.mean(),
    "any_link_%":    100*g.txt.str.contains("http").mean(),
}), include_groups=False).round(1).to_string())

print("\n\nThe four most-watched Shorts of 2025–26, verbatim\n")
for _, r in sh[sh.yr >= 2025].sort_values("views", ascending=False).head(4).iterrows():
    t = r.txt
    print(f"--- {r.channel} · {r.date.date()} · {r.views:,.0f} views · {len(t)} chars")
    print(f"    {t[:170]!r}\n")

> **This one holds, under the broad definition and by direct inspection.**
>
> Median Shorts description: **528 characters in 2024, 36 in 2025, 60 in 2026.** A quarter of 2026
> Shorts have a description of fewer than five characters. Storefront presence falls from 58% to
> **0.9%**, and that is with `lmg.gg` paths and the phrase "shop our products" included.
>
> The verbatim samples settle it. A Short with **42 million views** carries a 232-character prose
> caption and no links at all. One with **24 million views** has an empty description.
>
> So the surviving finding is narrower and better specified than what `06` claimed. It is not that
> the company is withdrawing from its storefront — long-form attachment is 99% and every long-form
> upload carries commercial links. It is that **the format now carrying a large and growing share
> of the network's reach has no commercial surface attached to it at all**, because its description
> is being written as a caption rather than as a set of links.

In [ ]:
tot = m[m.yr.between(2024, 2026)].groupby("yr").views.sum()
s   = sh[sh.yr.between(2024, 2026)].groupby("yr").views.sum()
print("Shorts share of all network views\n")
for y in [2024, 2025, 2026]:
    print(f"  {y}   Shorts {s.get(y,0)/1e6:7.1f}M of {tot[y]/1e6:7.1f}M  =  {s.get(y,0)/tot[y]:5.1%}")

print("\nBy channel, 2026 so far — where the gap actually is\n")
s26 = sh[sh.yr == 2026]
for ch, g in s26.groupby("channel"):
    gain = g.views.sum()*0.571 - g.loc[g.store_broad, "views"].sum()
    print(f"  {ch:15s} {len(g):3d} uploads · {g.views.sum()/1e6:6.1f}M views · "
          f"attachment {g.store_broad.mean():5.1%} · restoring 57% covers +{gain/1e6:5.1f}M")

> **The lever is on the flagship, not the small channels.** In the first half of 2026 the
> flagship published 54 Shorts that drew **121 million views** with no store link. ShortCircuit
> published 37 for 16 million. Restoring the 57% attachment the company itself ran for four years
> covers **+69 million** on the flagship against +9 million on ShortCircuit.
>
> Two things make this worth acting on rather than merely noting. It is a **format-level template
> change, not a strategy** — the same descriptions on long-form still carry the full link block. And
> it is reversible for the cost of editing a template.

### The corrected picture, in one figure

`06` produced a chart titled *"Store exposure fell 45%, not 29%"* with a corner decomposition
reading *reach −29% × attachment −12%*. Both halves of that were wrong, and the chart is not
salvageable by editing its caption — the series it plotted were domain-string matches, so the lines
themselves are artefacts.

This replaces it. Left: what the narrow domain match saw against what a census-built taxonomy sees.
Right: attachment by format, which is where the real gap is.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13.4, 4.6))

# ---- left: narrow vs broad, long-form -------------------------------------------
# rebuild explicitly: `lv` was rebound to the dv-based frame in section 5
lvm = m[m.format == "long_video"]
lvy = lvm[lvm.yr.between(2021, 2026)].groupby("yr")[["aff_narrow","aff_broad","store_broad"]].mean()*100
ax[0].plot(lvy.index, lvy.aff_broad,  "o-", color="#2E86DE", lw=2.4, ms=6,
           label="any commercial link — census-built taxonomy")
ax[0].plot(lvy.index, lvy.store_broad,"s--", color="#1B2430", lw=1.6, ms=5,
           label="storefront link")
ax[0].plot(lvy.index, lvy.aff_narrow, "o-", color="#E74C3C", lw=2.4, ms=6,
           label="what the original four-domain regex saw")
ax[0].set_ylim(-4, 108); ax[0].set_ylabel("Share of long-form uploads (%)")
ax[0].set_title("The surface never emptied — the strings moved", loc="left", weight="bold")
ax[0].legend(frameon=False, fontsize=8.6, loc="lower left")
ax[0].text(.98,.64,"Outbound links moved onto the company's own\n"
                   "shortener, lmg.gg, present in 93.5% of descriptions.\n"
                   "Matching literal domains read that as disappearance.",
           transform=ax[0].transAxes, ha="right", va="top", fontsize=8.4, color="#6B7684")

# ---- right: every surface, every year, on ShortCircuit long-form -----------------
sc = m[(m.channel == "ShortCircuit") & (m.format == "long_video") & m.yr.between(2021, 2026)].copy()
sc["storefront"]           = sc.store_broad
sc["affiliate / sponsor"]  = sc.aff_broad
sc["own paywall"]          = sc.txt.str.contains(r"(?i)floatplane", na=False)
sc["own shortener"]        = sc.txt.str.contains(r"lmg\.gg", case=False, na=False)
SURF = ["storefront", "affiliate / sponsor", "own paywall", "own shortener"]
cov  = sc.groupby("yr")[SURF].mean()*100

wid, xs = .2, np.arange(len(cov))
cols = ["#1B2430", "#2E86DE", "#7FB2E5", "#B9CFE8"]
for j, (s_, c_) in enumerate(zip(SURF, cols)):
    ax[1].bar(xs + (j-1.5)*wid, cov[s_], wid, color=c_, label=s_)
ax[1].set_xticks(xs); ax[1].set_xticklabels(cov.index)
ax[1].set_ylim(0, 122); ax[1].set_yticks([0, 25, 50, 75, 100])
ax[1].set_ylabel("Share of ShortCircuit long-form uploads (%)")
ax[1].set_title("Four surfaces, all full, every year", loc="left", weight="bold")
ax[1].legend(frameon=False, fontsize=8.4, ncol=2, loc="upper center")
for a in ax:
    a.spines[["top","right"]].set_visible(False); a.grid(alpha=.22)
    a.set_xlabel("")
plt.tight_layout(); plt.savefig(FIG/"commercial_surface.png", bbox_inches="tight"); plt.show()

In [ ]:
# ── for Remotion ──────────────────────────────────────────────────────────────
# Exports two things per figure: a transparent PLATE (axes, ticks, labels, titles,
# reference lines — no data marks) and a JSON of the series plus the geometry needed
# to map data coordinates onto the plate in pixels.
#
# The plate is rebuilt from scratch rather than stripped from the original, so a
# reference line can never be removed by accident. Numbers still come only from here.
REM = FIG/"remotion"; REM.mkdir(exist_ok=True)

def plate_geometry(fig, axes):
    """Pixel box of each axes plus its data limits — everything Remotion needs."""
    w_px, h_px = fig.get_size_inches()*fig.dpi
    out = []
    for a in axes:
        p = a.get_position()
        out.append({
            "px": {"x": p.x0*w_px, "y": (1-p.y1)*h_px, "w": p.width*w_px, "h": p.height*h_px},
            "xlim": list(a.get_xlim()), "ylim": list(a.get_ylim()),
            "xscale": a.get_xscale(), "yscale": a.get_yscale(),
        })
    return {"figure_px": {"w": w_px, "h": h_px}, "dpi": fig.dpi, "axes": out}

def save_plate(fig, name, series):
    fig.savefig(REM/f"{name}_plate.png", bbox_inches=None, transparent=True)
    geo = plate_geometry(fig, fig.axes)
    (REM/f"{name}.json").write_text(json.dumps({**geo, "series": series}, indent=1))
    print(f"  {name}_plate.png  +  {name}.json   "
          f"({geo['figure_px']['w']:.0f}x{geo['figure_px']['h']:.0f}px, {len(series)} series)")

# ---- commercial_surface, left panel: three lines over six years -------------------
lvm = m[m.format == "long_video"]
lvy = lvm[lvm.yr.between(2021, 2026)].groupby("yr")[["aff_narrow","aff_broad","store_broad"]].mean()*100
SER = [("aff_broad","#2E86DE","any commercial link — census-built taxonomy"),
       ("store_broad","#1B2430","storefront link"),
       ("aff_narrow","#E74C3C","what the original four-domain regex saw")]
series = [{"name": lab, "color": col, "kind": "line",
           "x": [float(v) for v in lvy.index], "y": [float(v) for v in lvy[k]]}
          for k, col, lab in SER]
from matplotlib.lines import Line2D
fig, ax = plt.subplots(figsize=(6.7, 4.6))
ax.plot(lvy.index, lvy.aff_broad, alpha=0)
STY = {"aff_broad": ("o-", 2.4), "store_broad": ("s--", 1.6), "aff_narrow": ("o-", 2.4)}
ax.legend(handles=[Line2D([], [], color=col, lw=STY[k][1],
                          ls="--" if k == "store_broad" else "-",
                          marker="s" if k == "store_broad" else "o", ms=5, label=lab)
                   for k, col, lab in SER], frameon=False, fontsize=8.6, loc="lower left")
ax.set_ylim(-4, 108); ax.set_ylabel("Share of long-form uploads (%)")
ax.set_title("The surface never emptied — the strings moved", loc="left", weight="bold")
ax.text(.98,.64,"Outbound links moved onto the company's own\n"
                "shortener, lmg.gg, present in 93.5% of descriptions.\n"
                "Matching literal domains read that as disappearance.",
        transform=ax.transAxes, ha="right", va="top", fontsize=8.4, color="#6B7684")
ax.spines[["top","right"]].set_visible(False); ax.grid(alpha=.22)
plt.tight_layout()
save_plate(fig, "commercial_surface_left", series); plt.close(fig)

> **Read the left panel as a warning about method, not about the company.** The red line is what a
> four-domain regex saw; the blue line is what is actually there. Every commercial claim in `06`
> that rested on the red line is void.
>
> **The right panel is the finding that survives.** Long-form attachment has been at or near 100%
> throughout — there is no room to add links to the format that carries the reviews. Shorts went
> the other way: from 58.8% attachment down to zero, while growing from 2% of the network's views
> to 41%. The description on a Short is now a caption, not a link block.

## 5. What this episode should change about the method

Four errors in this project now share a shape, and this one is the clearest.

- `02`: a 45-minute duration filter, described as neutral, that removed 15.5% of one channel and
  nothing from four others.
- `05`: a cluster read as a company-wide pivot to AI coverage that was one channel reporting news.
- `06`: blended attachment rate read as a behaviour change when it was Shorts growing as a share.
- `08`: a domain string read as a monetisation surface when the string had migrated to a shortener.

The first three are composition errors — averaging over a mix that was itself moving. **The fourth
is a proxy error, and it is more dangerous**, because a composition error usually leaves the total
looking odd, while a proxy error produces a clean, dramatic, entirely false result. The Floatplane
"single-month cliff" was the most striking chart in `06`. It was measuring a URL rewrite.

The rule this yields is specific enough to apply mechanically:

> **When a measurement is a string match, the first test is not "is the trend real" but "is the
> string still the same string".** Census the raw field before choosing patterns, and re-census
> whenever a series breaks sharply. A discontinuity in a text-derived metric is a schema change
> until proven otherwise.

The cost of not doing this was three findings and one recommendation, all of which I had already
reported with confidence. The thing that caught it was not a statistical check. **It was opening
one video and reading the description.**

## 6 · The figure the deck actually uses for slide 5

The two-panel figure above is the analytical record and stays. It is the wrong picture for a talk:
its left panel's red line — what the four-domain regex saw — falls to 21% while the narration says
"there is no room left to add links". Anyone not reading the caption sees the screen contradict the
sentence.

What slide 5 needs is one claim, shown once: **four commercial surfaces, six years, all of them at
the ceiling.** Grouped bars, because the surfaces are not mutually exclusive.

In [ ]:
# ── Slide 5's figure: the four commercial surfaces, by year ───────────────────
# Written self-contained (it rebuilds `m` from disk) so it can be re-run without
# executing the census above it.
#
# GROUPED bars, never stacked. The four surfaces are not mutually exclusive — one
# upload carries the storefront AND an affiliate link AND Floatplane AND the
# shortener — so a stack would sum to ~400% and an overlay would hide three bars
# behind the tallest. Grouped is the only honest arrangement, and bars at the
# ceiling say "full" in a way a flat line does not.
#
# Every pattern is the census-built one from section 3, not the four-domain regex
# that section 2 showed to be an artefact of the lmg.gg migration.
import json, pathlib, pandas as pd

DATA = pathlib.Path("data"); REM = pathlib.Path("figures/remotion"); REM.mkdir(parents=True, exist_ok=True)
desc = pd.DataFrame([json.loads(l) for l in open(DATA/"content/descriptions.jsonl") if l.strip()])
vids = pd.read_csv(DATA/"videos_clean.csv", parse_dates=["date"])
mm   = vids.merge(desc, on="id")
col  = "description" if "description" in mm.columns else [c for c in mm.columns if "desc" in c][0]
mm["txt"] = mm[col].fillna(""); mm["yr"] = mm.date.dt.year
lvs = mm[(mm.format == "long_video") & mm.yr.between(2021, 2026)].copy()

SURFACES = {
    "storefront":     r"lttstore\.com|lmg\.gg/\S*(?:store|merch|shop)",
    "affiliate":      (r"amzn\.to|geni\.us|howl\.link|tag=|affilimate\.com|nexus\.gg|kit\.co|bit\.ly|"
                       r"piavpn\.com|lmg\.gg/\S*(?:partner|sponsor)|shortlinus\.com|dbrand\.com|"
                       r"vessi\.com|ridge\.com|squarespace\.com|manscaped\.com|joindeleteme\.com|"
                       r"micro\.center|freshbooks\.com|odoo\.com|grammarly\.com|buildredux\.com|"
                       r"dro\.ps|prsm2\.com|playwt\.link|tm\.game"),
    "Floatplane":     r"floatplane\.com|lmg\.gg/\S*floatplane",
    "own shortener":  r"lmg\.gg/",
}
for k, pat in SURFACES.items():
    lvs[k] = lvs.txt.str.contains(pat, case=False, na=False, regex=True)

tab = (lvs.groupby("yr")[list(SURFACES)].mean() * 100).round(2)
print("Long-form uploads carrying each commercial surface, %\n")
print(tab.round(1).to_string())
print("\nuploads per year:", lvs.groupby("yr").size().to_dict())
print(f"at least one of the four: {lvs[list(SURFACES)].any(axis=1).mean()*100:.2f}%")
print(f"lowest single cell in the whole table: {tab.values.min():.1f}%")

# Bar geometry travels with the data so Remotion never guesses it.
GROUP, N = 0.86, len(SURFACES)
BW = GROUP / N
offsets = [-GROUP/2 + BW*(i + .5) for i in range(N)]
COLORS = ["#4DA3FF", "#7FB2E5", "#E8ECF2", "#1E6FB8"]   # one hue family: all four are "full"
out = {"bar_width": BW, "series": [
    {"name": k, "color": COLORS[i], "kind": "bar",
     "x": [int(y) + offsets[i] for y in tab.index], "y": [float(v) for v in tab[k]],
     "year": [int(y) for y in tab.index]}
    for i, k in enumerate(SURFACES)]}
json.dump(out, open(REM/"commercial_surfaces.json", "w"), indent=1)
print(f"\nwrote {REM/'commercial_surfaces.json'} · {N} series x {len(tab)} years, bar width {BW:.3f}")


Long-form uploads carrying each commercial surface, %

      storefront  affiliate  Floatplane  own shortener
yr                                                    
2021        99.1      100.0        99.1           99.5
2022        99.5       90.4        99.5           99.5
2023        98.9       99.0        98.6           99.6
2024        98.8       99.4        98.8           99.7
2025        99.5       99.8        99.3           99.5
2026       100.0      100.0       100.0          100.0

uploads per year: {2021: 757, 2022: 784, 2023: 699, 2024: 661, 2025: 564, 2026: 313}
at least one of the four: 99.89%
lowest single cell in the whole table: 90.4%

wrote figures/remotion/commercial_surfaces.json · 4 series x 6 years, bar width 0.215
